# 02 · Forward pass and decoding
Load FlyNet / CTRNN, run a stimulus, decode hit/miss and position/velocity. **Requires** `flyvis`, `pytorch3d`.

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from flynet_robustness.models import get_flynet, get_stimulus_and_return_activity
from flynet_robustness.decoders import load_decoder, load_decoder_pos_vel
from flynet_robustness.textures import get_texture
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# ids: 'FlyNet', 'CTRNN', 'sCTRNN', 'flynet_degree_matched_0' ... 'flynet_degree_matched_9'
model1, model2, idxs1, idxs2 = get_flynet('FlyNet', device=device)
print('decoded T4/T5 units:', len(idxs1))

In [ ]:
activity, speed, angle, position = get_stimulus_and_return_activity(
    model1, idxs1, model2, idxs2, hit=True, shape='Sphere',
    texture=get_texture('Black'), time=1.0, device=device)
print('activity', tuple(activity.shape))

## Hit/miss probability

In [ ]:
from flynet_robustness.plotting import plot_p_hit_trace
decoder = load_decoder('FlyNet')
p_hit = torch.sigmoid(decoder(activity.flatten(end_dim=1).to(device))).detach().cpu().numpy()
plot_p_hit_trace(p_hit, color='forestgreen');

## Position / velocity (trained regressors under `decoders/FlyNet/`)

In [ ]:
from flynet_robustness.plotting import plot_decoding_scatter
vel = load_decoder_pos_vel('FlyNet', load_velocity=True, device=device)
pred = vel(activity.flatten(end_dim=1).to(device)).detach().cpu().numpy()
print('predicted velocity', pred.shape)